# 05 - Features para tendência diária e semanal

Até o notebook 04, o projeto responde o que aconteceu com as ações. A partir daqui, a pergunta muda: **o próximo período tende a fechar em alta ou em baixa?**

Vou trabalhar somente com informações derivadas do preço de fechamento. O objetivo é criar duas bases para classificação:

- tendência do próximo pregão;
- tendência da próxima semana.

O target será binário: `1` para alta e `0` para baixa ou estabilidade. Isso não representa uma ordem de compra ou venda. Primeiro os modelos precisam superar um baseline em dados futuros.

In [ ]:
from pyspark.sql import functions as F, Window

df_silver = spark.table("b3_pipeline.silver_b3_stocks")
df_silver.printSchema()

## Parte 1 - Features diárias

Cada ticker é processado separadamente e em ordem cronológica. As features usam somente o fechamento atual e valores passados:

- retorno diário;
- médias móveis de 5 e 20 pregões;
- distância do fechamento para cada média;
- momentum de 5 e 20 pregões;
- volatilidade dos retornos em 5 e 20 pregões;
- retornos defasados em 1, 2 e 5 pregões.

In [ ]:
w_dia = Window.partitionBy("ticker").orderBy("date")
w_5d = w_dia.rowsBetween(-4, 0)
w_20d = w_dia.rowsBetween(-19, 0)

df_diario = df_silver \
    .withColumn("media_close_5d", F.avg("close").over(w_5d)) \
    .withColumn("media_close_20d", F.avg("close").over(w_20d)) \
    .withColumn("volatilidade_5d", F.stddev("daily_return_pct").over(w_5d)) \
    .withColumn("volatilidade_20d", F.stddev("daily_return_pct").over(w_20d)) \
    .withColumn("close_lag5", F.lag("close", 5).over(w_dia)) \
    .withColumn("close_lag20", F.lag("close", 20).over(w_dia)) \
    .withColumn("retorno_lag1", F.lag("daily_return_pct", 1).over(w_dia)) \
    .withColumn("retorno_lag2", F.lag("daily_return_pct", 2).over(w_dia)) \
    .withColumn("retorno_lag5", F.lag("daily_return_pct", 5).over(w_dia)) \
    .withColumn("distancia_media_5d_pct", (F.col("close") / F.col("media_close_5d") - 1) * 100) \
    .withColumn("distancia_media_20d_pct", (F.col("close") / F.col("media_close_20d") - 1) * 100) \
    .withColumn("momentum_5d_pct", (F.col("close") / F.col("close_lag5") - 1) * 100) \
    .withColumn("momentum_20d_pct", (F.col("close") / F.col("close_lag20") - 1) * 100)

## Target diário

`next_close` traz o fechamento do próximo pregão. Se ele for maior que o fechamento atual, o target recebe `1`; caso contrário, recebe `0`.

A última linha de cada ticker fica sem target, porque o próximo fechamento ainda não existe. Ela será preservada para uma futura estimativa, mas não entra no treino.

In [ ]:
df_diario = df_diario \
    .withColumn("next_close", F.lead("close", 1).over(w_dia)) \
    .withColumn("target_retorno_prox_dia_pct", (F.col("next_close") / F.col("close") - 1) * 100) \
    .withColumn(
        "target_tendencia_prox_dia",
        F.when(F.col("next_close").isNull(), F.lit(None))
         .when(F.col("next_close") > F.col("close"), F.lit(1))
         .otherwise(F.lit(0))
         .cast("int")
    ) \
    .select(
        "ticker", "setor", "date", "close", "daily_return_pct",
        "media_close_5d", "media_close_20d",
        "distancia_media_5d_pct", "distancia_media_20d_pct",
        "momentum_5d_pct", "momentum_20d_pct",
        "volatilidade_5d", "volatilidade_20d",
        "retorno_lag1", "retorno_lag2", "retorno_lag5",
        "target_retorno_prox_dia_pct", "target_tendencia_prox_dia"
    )

df_diario.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_tendencia_features_diario")

print("Features diárias salvas com sucesso!")
display(df_diario.orderBy(F.desc("date")))

## Parte 2 - Fechamento semanal

Agora agrupo os pregões por semana. Para cada ticker, `max_by` pega o fechamento correspondente à data mais recente daquela semana.

A frequência semanal reduz parte do ruído diário e permite testar a mesma pergunta em outro horizonte.

In [ ]:
df_semanal_base = df_silver \
    .withColumn("week_start", F.to_date(F.date_trunc("week", F.col("date")))) \
    .groupBy("ticker", "setor", "week_start") \
    .agg(
        F.max("date").alias("week_last_date"),
        F.max_by("close", "date").alias("weekly_close")
    )

w_semana = Window.partitionBy("ticker").orderBy("week_start")
w_4s = w_semana.rowsBetween(-3, 0)
w_12s = w_semana.rowsBetween(-11, 0)

## Features e target semanais

A lógica é a mesma da base diária, mas as janelas agora representam semanas. O target recebe `1` quando o fechamento da próxima semana é maior que o fechamento atual.

In [ ]:
df_semanal = df_semanal_base \
    .withColumn("weekly_close_lag1", F.lag("weekly_close", 1).over(w_semana)) \
    .withColumn("weekly_return_pct", (F.col("weekly_close") / F.col("weekly_close_lag1") - 1) * 100) \
    .withColumn("media_close_4s", F.avg("weekly_close").over(w_4s)) \
    .withColumn("media_close_12s", F.avg("weekly_close").over(w_12s)) \
    .withColumn("volatilidade_4s", F.stddev("weekly_return_pct").over(w_4s)) \
    .withColumn("volatilidade_12s", F.stddev("weekly_return_pct").over(w_12s)) \
    .withColumn("close_lag4s", F.lag("weekly_close", 4).over(w_semana)) \
    .withColumn("close_lag12s", F.lag("weekly_close", 12).over(w_semana)) \
    .withColumn("retorno_lag1s", F.lag("weekly_return_pct", 1).over(w_semana)) \
    .withColumn("retorno_lag2s", F.lag("weekly_return_pct", 2).over(w_semana)) \
    .withColumn("retorno_lag4s", F.lag("weekly_return_pct", 4).over(w_semana)) \
    .withColumn("distancia_media_4s_pct", (F.col("weekly_close") / F.col("media_close_4s") - 1) * 100) \
    .withColumn("distancia_media_12s_pct", (F.col("weekly_close") / F.col("media_close_12s") - 1) * 100) \
    .withColumn("momentum_4s_pct", (F.col("weekly_close") / F.col("close_lag4s") - 1) * 100) \
    .withColumn("momentum_12s_pct", (F.col("weekly_close") / F.col("close_lag12s") - 1) * 100) \
    .withColumn("next_week_close", F.lead("weekly_close", 1).over(w_semana)) \
    .withColumn("target_retorno_prox_semana_pct", (F.col("next_week_close") / F.col("weekly_close") - 1) * 100) \
    .withColumn(
        "target_tendencia_prox_semana",
        F.when(F.col("next_week_close").isNull(), F.lit(None))
         .when(F.col("next_week_close") > F.col("weekly_close"), F.lit(1))
         .otherwise(F.lit(0))
         .cast("int")
    ) \
    .select(
        "ticker", "setor", "week_start", "week_last_date", "weekly_close", "weekly_return_pct",
        "media_close_4s", "media_close_12s",
        "distancia_media_4s_pct", "distancia_media_12s_pct",
        "momentum_4s_pct", "momentum_12s_pct",
        "volatilidade_4s", "volatilidade_12s",
        "retorno_lag1s", "retorno_lag2s", "retorno_lag4s",
        "target_retorno_prox_semana_pct", "target_tendencia_prox_semana"
    )

df_semanal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.gold_tendencia_features_semanal")

print("Features semanais salvas com sucesso!")
display(df_semanal.orderBy(F.desc("week_start")))

## Resultado

O notebook gera duas tabelas Gold:

- `b3_pipeline.gold_tendencia_features_diario`;
- `b3_pipeline.gold_tendencia_features_semanal`.

Os notebooks 06 e 07 usarão somente linhas com features completas e target conhecido para treinar os classificadores. As linhas mais recentes, ainda sem target, serão usadas para gerar uma estimativa educacional de tendência.